In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import json
from matplotlib import cm
from collections import defaultdict

In [4]:
def parse_fasta_headers(fasta_file):
    """
    解析FASTA文件，建立subject_id到FASTA头的映射
    
    参数:
        fasta_file: FASTA文件路径
        
    返回:
        header_dict: 字典，键为subject_id，值为FASTA头信息字典
    """
    header_dict = {}
    current_header = ""
    subject_id = ""
    
    with open(fasta_file, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                # 处理头部行
                current_header = line[1:]  # 去掉'>'符号
                
                # 提取subject_id（第一个空格前的部分）
                subject_id = current_header.split()[0]
                
                # 尝试解析JSON部分
                json_start = current_header.find('{')
                if json_start != -1:
                    json_str = current_header[json_start:]
                    try:
                        header_info = json.loads(json_str)
                        header_dict[subject_id] = header_info
                    except json.JSONDecodeError:
                        print(f"警告: 无法解析JSON: {json_str}")
                        header_dict[subject_id] = {"raw_header": current_header}
                else:
                    header_dict[subject_id] = {"raw_header": current_header}
    
    return header_dict

def extract_genus(organism_name):
    """
    从organism_name中提取属名
    
    参数:
        organism_name: 完整的物种名称
        
    返回:
        属名
    """
    if not organism_name:
        return "Unknown"
    
    # 取第一个单词作为属名
    return organism_name.split()[0]

In [ ]:
# 存储数据的列表
def process_blast_with_fasta_mapping(blast_file, fasta_file):
    """
    处理BLAST结果，并使用FASTA文件中的信息进行丰富
    
    参数:
        blast_file: BLAST结果文件路径
        fasta_file: FASTA文件路径
    """
    # 首先解析FASTA文件建立映射
    header_dict = parse_fasta_headers(fasta_file)
    
    # 存储BLAST结果的数据
    blast_data = []
    
    # 读取BLAST结果
    with open(blast_file, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 12:
                continue
                
            subject_id = parts[1]  # 第二列是subject_id
            identity = float(parts[2])  # 第三列是比对率
            p_value = float(parts[10])  # 第十一列是p值
            
            # 获取FASTA头信息
            header_info = header_dict.get(subject_id, {})
            organism_name = header_info.get("organism_name", "Unknown")
            genus = extract_genus(organism_name)
            
            blast_data.append({
                'query_id': parts[0],
                'subject_id': subject_id,
                'identity': identity,
                'p_value': p_value,
                'organism_name': organism_name,
                'genus': genus,
                'full_header_info': header_info
            })
    
    # 创建DataFrame
    df = pd.DataFrame(blast_data)
    
    return df

# 使用示例
df = process_blast_with_fasta_mapping("rv0047c.txt", "fasta.fasta")

# 现在df包含所有需要的信息，可以用于绘制曼哈顿图
print(df.head())

In [ ]:
def plot_enhanced_grouped_barplot_with_legend(df):
    """
    增强版分组柱状图，显示属名称标签和颜色图例，添加中位数虚线
    """
    # 计算-log10p值
    df['neg_log10_p'] = -np.log10(df['p_value'])
    df = df.sort_values('genus')
    
    # 创建图形
    fig, ax = plt.subplots(figsize=(25, 10))
    
    # 计算中位数
    median_value = df['neg_log10_p'].median()
    
    # 绘制柱状图
    x_positions = range(len(df))
    
    # 为每个属分配颜色（使用tab20颜色映射）
    unique_genera = df['genus'].unique()
    colors = plt.cm.tab20(np.linspace(0, 1, min(20, len(unique_genera))))
    
    # 如果属数量超过20个，使用Set3继续分配颜色
    if len(unique_genera) > 20:
        additional_colors = plt.cm.Set3(np.linspace(0, 1, min(12, len(unique_genera) - 20)))
        colors = np.concatenate([colors, additional_colors])
    
    # 如果还不够，循环使用颜色
    color_dict = {}
    for i, genus in enumerate(unique_genera):
        color_dict[genus] = colors[i % len(colors)]
    
    # 绘制柱子
    bars = ax.bar(x_positions, df['neg_log10_p'], 
                 width=1.0,
                 color=[color_dict[genus] for genus in df['genus']],
                 alpha=0.8)
    
    # 添加中位数虚线
    ax.axhline(y=median_value, color='#B02425', linestyle='--', linewidth=2, 
               alpha=0.8)
    
    # 在中位数虚线末尾上方添加数值标签
    ax.text(x=len(df) * 1.045,  # x位置：图表宽度的98%处（右侧）
            y=median_value * 1.02,  # y位置：中位数上方2%处
            s=f'Median: {median_value:.2f}',
            color='black',
            fontsize=7,
            fontweight='bold',
            ha='right',  # 水平对齐：右对齐
            va='bottom')  # 垂直对齐：底部对齐
    
    # 添加颜色图例
    legend_elements = []
    for genus in unique_genera:
        legend_elements.append(plt.Rectangle((0, 0), 1, 1, 
                                           facecolor=color_dict[genus],
                                           edgecolor='black',
                                           label=genus))
    
    # 创建图例（根据属的数量调整列数）
    ncol = 3 if len(unique_genera) > 15 else 2
    if len(unique_genera) > 30:
        ncol = 4
    
    legend = ax.legend(handles=legend_elements, 
                      loc='upper center', 
                      bbox_to_anchor=(0.5, -0.15),
                      ncol=ncol,
                      fontsize=8,
                      frameon=True,
                      fancybox=True,
                      shadow=True)
    
    # 设置图例标题
    legend.set_title('Genus', prop={'size': 10, 'weight': 'bold'})
    
    ax.set_xlabel('Protein Sequence Index')
    ax.set_ylabel('-log10(p-value)')
    ax.set_title('Grouped Bar Plot of -log10(p-value) by Genus with Color Legend')
    ax.grid(axis='y', alpha=0.3)
    
    # 由于点数太多，隐藏x轴刻度
    ax.set_xticks([])
    
    # 调整布局，为图例留出空间
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.2)  # 为底部图例留出空间
    plt.savefig('log-10.pdf', 
           bbox_inches='tight',  # 自动调整边界，确保所有内容都包含在内
           dpi=10000,              # 提高分辨率
           format='pdf')         # 明确指定格式
    plt.show()
    
    return color_dict

# 使用带图例的增强版
color_mapping = plot_enhanced_grouped_barplot_with_legend(df)